# Previsão de Household Income por município (até 2026)

**Objetivo**: a partir do ficheiro `household-income-15-21.parquet`, eu projeto valores de *household income* para 2022–2026, município a município, usando 5 componentes: (1) tendência local (CAGR pré-COVID), (2) inflação observada, (3) crescimento do PIB, (4) ajuste territorial (cluster+convergência) e (5) crescimento salarial (mínimo + proxy mediano).

**Nota**: este notebook foi gerado em 2026-05-16.

## Fontes (para defender o trabalho)

Eu não faço *forecast* da inflação/salário mínimo: eu uso séries publicadas.

- Salário mínimo Mensal nacional (tabela anual, inclui 2026): https://www.pordata.pt/portugal/evolucao+do+salario+minimo+nacional-74

- Salário Médio Líquido Mensal Em portugal - https://pt.tradingeconomics.com/portugal/wages

- Previsão macro (PIB) – PORDATA - https://www.pordata.pt/portugal/taxa+de+crescimento+do+pib-2298
    - Estimativa para 2026 https://www.bportugal.pt/publicacao/boletim-economico-marco-2026

-Inflação segundo Banco de Portugal - https://bpstat.bportugal.pt/serie/12704650
    - Estimativa para 2026: https://pt.tradingeconomics.com/portugal/inflation-cpi


- (Proxy para mediana) ‘Rendimento mediano disponível’ na PORDATA Retratos da Europa: https://retratos.europa.pordata.pt/custo-de-vida/portugal

⚠️ **Sobre salário mediano**: se não tiveres uma série direta de *salário mediano*, eu uso como proxy o **rendimento mediano disponível** (mais alinhado com household). Se encontrares uma série de salário mediano, basta substituir a tabela `median_proxy`.

In [2]:
# Imports
import pandas as pd
import numpy as np


## 0) Ler o dataset e normalizar colunas

Aqui eu leio o parquet e normalizo nomes/Tipos para evitar erros a meio do pipeline.

In [3]:
# Eu leio o dataset parquet para um DataFrame.
df = pd.read_parquet('household-income-15-21.parquet')

# Eu normalizo nomes de colunas para evitar acentos/espaços.
df.columns = (
    df.columns.astype(str)
      .str.strip()
      .str.replace(' ', '_')
      .str.replace('(', '', regex=False)
      .str.replace(')', '', regex=False)
      .str.lower()
)

# Eu tento mapear para os nomes canónicos year/municipio/income.
rename_map = {
    'ano': 'year',
    'município': 'municipio',
    'municipio': 'municipio',
    'household_income_(eur)': 'income',
    'household_income_eur': 'income',
    'household_income': 'income'
}

for k, v in rename_map.items():
    if k in df.columns:
        df = df.rename(columns={k: v})

# Eu valido que tenho as colunas mínimas.
required = {'year', 'municipio', 'income'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Faltam colunas no parquet: {missing}. Colunas existentes: {list(df.columns)}')

# Eu asseguro tipos.
df['year'] = df['year'].astype(int)
df['municipio'] = df['municipio'].astype(str)
df['income'] = pd.to_numeric(df['income'], errors='coerce')

# Eu ordeno por municipio/ano para cálculos temporais.
df = df.sort_values(['municipio', 'year']).reset_index(drop=True)

df.head()

,year,municipio,income
0,2015,Abrantes,15108
1,2016,Abrantes,15609
2,2017,Abrantes,16014
3,2018,Abrantes,16574
4,2019,Abrantes,17115


## 1) Calcular CAGR pré-COVID (até 2019)

Aqui eu calculo a taxa anual composta de crescimento usando só anos até 2019. Eu faço isto para captar a tendência estrutural e reduzir distorções de 2020–2021.

In [4]:
def calculate_pre_covid_cagr(group: pd.DataFrame) -> float:
    # Eu filtro até 2019 para estimar crescimento estrutural.
    g = group[group['year'] <= 2019].copy()

    # Eu preciso de pelo menos 2 pontos.
    if g.shape[0] < 2:
        return np.nan

    start = g.iloc[0]['income']
    end = g.iloc[-1]['income']
    years = g.iloc[-1]['year'] - g.iloc[0]['year']

    # Eu protejo contra zeros/NaNs.
    if pd.isna(start) or pd.isna(end) or start <= 0 or years <= 0:
        return np.nan

    return (end / start) ** (1 / years) - 1

cagr_df = (
    df.groupby('municipio', as_index=False)
      .apply(lambda g: pd.Series({'cagr_pre_covid': calculate_pre_covid_cagr(g)}))
)

cagr_df.head()

C:\Users\luisa\AppData\Local\Temp\ipykernel_16896\154832965.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({'cagr_pre_covid': calculate_pre_covid_cagr(g)}))


,municipio,cagr_pre_covid
0,Abrantes,0.031674
1,Aguiar da Beira,0.056820
2,Alandroal,0.048184
3,Albergaria-a-Velha,0.034377
4,Albufeira,0.034842


## 2) Construir a tabela macro (inflação, PIB, salários)

Aqui eu defino as séries anuais para 2022–2026.

- Eu **não puxo automaticamente** da internet porque o ambiente do notebook pode não ter acesso; em vez disso, eu deixo uma tabela simples para preencher, baseada nos links do topo.
- Eu valido que todos os anos necessários existem, para evitar correr o modelo com buracos.

**Como preencher**: copia os valores dos sites e substitui nos dicionários abaixo.

In [6]:
# ======== CONFIG (PREENCHER) ========
YEARS_FORECAST = [2022, 2023, 2024, 2025, 2026]

# (2) Inflação anual (% em decimal). Fonte recomendada: INE (IPC) / PORDATA / ECB.
# Exemplo: 2.4% -> 0.024
inflation = {
    2022: 0.0783,
    2023: 0.0431,
    2024: 0.0242,
    2025: 0.0234,
    2026: 0.0330,
}

# (3) Crescimento real do PIB (% em decimal). Fonte: Comissão Europeia / Banco de Portugal / OCDE.
gdp_growth = {
    2022: 0.07,
    2023: 0.0310,
    2024: 0.022,
    2025: 0.019,
    2026: 0.018,
}

# (5) Salário mínimo nacional (valor mensal, EUR). Fonte: PORDATA ou DGERT.
min_wage = {
    2022: 705,
    2023: 760,
    2024: 820,
    2025: 870,
    2026: 820,
}

# (5) Proxy mediano (preferido): rendimento mediano disponível (EUR/mês) ou salário mediano se tiveres.
# Fonte proxy (PORDATA retratos): rendimento mediano disponível.
median_proxy = {
    2022: 1010,
    2023: 1090,
    2024: 1230,
    2025: 1310,
    2026: 1370,
}

# Pesos para combinar mínimo + proxy mediano num único 'driver salarial'.
# Eu dou mais peso ao proxy mediano (mais alinhado com household).
W_MIN = 0.30
W_MED = 0.70

# ======== FIM CONFIG ========

def _validate_years(d: dict, name: str, years=YEARS_FORECAST):
    missing = [y for y in years if y not in d]
    if missing:
        raise ValueError(f'Faltam anos em {name}: {missing}')
    nan_years = [y for y in years if pd.isna(d[y])]
    if nan_years:
        raise ValueError(
            f'Faltam valores (NaN) em {name} para anos: {nan_years}. ' 
            f'Preenche estes valores com base nas fontes do topo.'
        )

_validate_years(inflation, 'inflation')
_validate_years(gdp_growth, 'gdp_growth')
_validate_years(min_wage, 'min_wage')
_validate_years(median_proxy, 'median_proxy')

macro = pd.DataFrame({
    'year': YEARS_FORECAST,
    'inflation': [inflation[y] for y in YEARS_FORECAST],
    'gdp_growth': [gdp_growth[y] for y in YEARS_FORECAST],
    'min_wage': [min_wage[y] for y in YEARS_FORECAST],
    'median_proxy': [median_proxy[y] for y in YEARS_FORECAST],
})

# Aqui eu converto salário mínimo e proxy mediano para taxas de crescimento anuais.
macro['g_min_wage'] = macro['min_wage'].pct_change()
macro['g_median_proxy'] = macro['median_proxy'].pct_change()

# Para 2022 (primeiro ano), eu não tenho variação anterior dentro do horizonte.
# Eu assumo 0 para não introduzir um salto artificial; se quiseres, podes calcular 2021->2022.
macro.loc[macro['year'] == YEARS_FORECAST[0], ['g_min_wage', 'g_median_proxy']] = 0.0

# Aqui eu agrego um driver salarial único.
macro['g_wage_signal'] = W_MIN * macro['g_min_wage'] + W_MED * macro['g_median_proxy']

macro

,year,inflation,gdp_growth,min_wage,median_proxy,g_min_wage,g_median_proxy,g_wage_signal
0,2022,0.0783,0.070,705,1010,0.000000,0.000000,0.000000
1,2023,0.0431,0.031,760,1090,0.078014,0.079208,0.078850
2,2024,0.0242,0.022,820,1230,0.078947,0.128440,0.113592
3,2025,0.0234,0.019,870,1310,0.060976,0.065041,0.063821
4,2026,0.0330,0.018,820,1370,-0.057471,0.045802,0.014820


## 3) Ajuste territorial (cluster + convergência)

Aqui eu junto num único passo aquilo que combinámos:

- Cluster urbano vs resto (ajuste leve)
- Convergência suave (municípios abaixo da média crescem % ligeiramente mais)

Eu mantenho isto simples para ser auditável e fácil de explicar.

In [7]:
def classify_cluster(municipio: str) -> str:
    # Eu começo com uma lista pequena e óbvia (podes expandir).
    urban = {"Lisboa", "Porto", "Cascais", "Oeiras", "Amadora", "Sintra"}
    return 'urban' if municipio in urban else 'other'

# Tabela de clusters por município
cluster_df = pd.DataFrame({
    'municipio': df['municipio'].unique(),
})
cluster_df['cluster'] = cluster_df['municipio'].apply(classify_cluster)

cluster_df.head()

KeyError: 'municipio'

In [ ]:
def adjust_growth(base_growth: float, cluster: str, current_income: float, national_avg: float) -> float:
    # Se eu não tiver CAGR, eu uso um fallback conservador.
    if pd.isna(base_growth):
        base_growth = 0.02

    # Ajuste por cluster (leve para não exagerar divergência).
    cluster_multiplier = 1.05 if cluster == 'urban' else 0.95

    # Convergência suave (alpha controla agressividade).
    alpha = 0.10
    convergence = alpha * (national_avg - current_income) / national_avg

    return base_growth * cluster_multiplier + convergence


## 4) Preparar base (último ano observado) e juntar features

Aqui eu construo o 'estado inicial' por município (último ano real) e junto CAGR + cluster.

In [ ]:
latest_year = int(df['year'].max())

base_df = df[df['year'] == latest_year].copy()

base_df = base_df.merge(cagr_df, on='municipio', how='left')
base_df = base_df.merge(cluster_df, on='municipio', how='left')

base_df.head()

## 5) Loop de projeção (2022–2026) e dataset final

Aqui eu projeto ano a ano, aplicando as 5 componentes:

1) CAGR local (pré-COVID)
2) inflação
3) PIB
4) territorial (cluster+convergência)
5) salários (mínimo + proxy mediano)

No fim, eu junto histórico + projeções e guardo em parquet.

In [ ]:
projections = []
current_df = base_df.copy()

for _, row in macro.sort_values('year').iterrows():
    year = int(row['year'])

    # Eu calculo a média nacional no estado atual (para convergência).
    national_avg = current_df['income'].mean()

    next_df = current_df.copy()

    # (1 + 4) Eu calculo crescimento local ajustado (CAGR + territorial).
    next_df['g_local_adj'] = next_df.apply(
        lambda r: adjust_growth(
            base_growth=r['cagr_pre_covid'],
            cluster=r['cluster'],
            current_income=r['income'],
            national_avg=national_avg
        ),
        axis=1
    )

    # (2) inflação observada
    g_infl = float(row['inflation'])

    # (3) PIB
    g_gdp = float(row['gdp_growth'])

    # (5) driver salarial agregado
    g_wage = float(row['g_wage_signal'])

    # Eu somo as componentes em taxa.
    next_df['g_total'] = next_df['g_local_adj'] + g_infl + g_gdp + g_wage

    # Eu aplico o crescimento ao income.
    next_df['income'] = next_df['income'] * (1 + next_df['g_total'])
    next_df['year'] = year

    projections.append(next_df[['municipio', 'year', 'income']])

    current_df = next_df

projection_df = pd.concat(projections, ignore_index=True)

final_df = pd.concat(
    [df[['municipio', 'year', 'income']], projection_df],
    ignore_index=True
).sort_values(['municipio', 'year']).reset_index(drop=True)

final_df.to_parquet('household_income_projection_2026.parquet', index=False)

final_df.head(20)

## 6) Sanity checks rápidos

Eu verifico rapidamente se os resultados fazem sentido: top municípios em 2026, e uma série exemplar (ex.: Lisboa).

In [ ]:
# Top 10 municípios em 2026
final_df[final_df['year'] == 2026].sort_values('income', ascending=False).head(10)


In [ ]:
# Exemplo: evolução de Lisboa (se existir)
if (final_df['municipio'] == 'Lisboa').any():
    display(final_df[final_df['municipio'] == 'Lisboa'].sort_values('year'))
else:
    print('Lisboa não existe na coluna municipio deste dataset.')
